In [ ]:
import sys

print("Python version:", sys.version)
print("Python executable:", sys.executable)

Python version: 3.13.16 (tags/v3.13.16:cbc944f, Sep 30 2026, 19:17:50) [MSC v.1944 64 bit (AMD64)]
Python executable: c:\Users\priya\Desktop\LLM\AI_Agent\.venv\Scripts\python.exe


In [3]:
%pip uninstall sqlalchemy -y

Found existing installation: SQLAlchemy 2.0.36
Uninstalling SQLAlchemy-2.0.36:
  Successfully uninstalled SQLAlchemy-2.0.36
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.


In [4]:
%pip install sqlalchemy==2.0.36

  Using cached SQLAlchemy-2.0.36-cp313-cp313-win_amd64.whl.metadata (9.9 kB)
Using cached SQLAlchemy-2.0.36-cp313-cp313-win_amd64.whl (2.1 MB)
Note: you may need to restart the kernel to use updated packages.


In [5]:
import sqlalchemy

print("SQLAlchemy version:", sqlalchemy.__version__)
print("✅ SQLAlchemy is working")

SQLAlchemy version: 2.0.36
✅ SQLAlchemy is working


In [6]:
import sqlite3
import os

from dotenv import load_dotenv
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_groq import ChatGroq
from langchain.agents import create_agent

print("✅ All imports successful")

C:\Users\priya\AppData\Local\Temp\ipykernel_15812\1636254002.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


✅ All imports successful


In [7]:
from pathlib import Path

env_path = Path.cwd() / ".env"

print("Current folder:", Path.cwd())
print(".env exists:", env_path.exists())
print(".env path:", env_path)

Current folder: c:\Users\priya\Desktop\LLM\AI_Agent
.env exists: True
.env path: c:\Users\priya\Desktop\LLM\AI_Agent\.env


In [8]:
from pathlib import Path

env_path = Path.cwd() / ".env"

env_path.write_text(
    "GROQ_API_KEY=PASTE_YOUR_GROQ_API_KEY_HERE\n",
    encoding="utf-8"
)

print("✅ .env file created")
print("Location:", env_path)

✅ .env file created
Location: c:\Users\priya\Desktop\LLM\AI_Agent\.env


In [9]:
load_dotenv(override=True)

groq_key = os.getenv("GROQ_API_KEY")

if not groq_key:
    raise ValueError("❌ GROQ_API_KEY not found")

print("✅ Groq API key loaded successfully")
print("Key length:", len(groq_key))


✅ Groq API key loaded successfully
Key length: 28


In [10]:
import sqlite3

connection = sqlite3.connect("example.db")
cursor = connection.cursor()

cursor.execute("DROP TABLE IF EXISTS inventory")

cursor.execute("""
CREATE TABLE inventory (
    id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    stock_quantity INTEGER NOT NULL
)
""")

products = [
    ("Product A", 150),
    ("Product B", 200),
    ("Product C", 300),
    ("Product D", 100),
    ("Product E", 50),
    ("Product F", 800)
]

cursor.executemany("""
INSERT INTO inventory (product_name, stock_quantity)
VALUES (?, ?)
""", products)

connection.commit()
connection.close()

print("✅ Database created successfully")
print("✅ Inventory table created")
print("✅ 6 products inserted")

✅ Database created successfully
✅ Inventory table created
✅ 6 products inserted


In [11]:
connection = sqlite3.connect("example.db")
cursor = connection.cursor()

cursor.execute("SELECT * FROM inventory")
rows = cursor.fetchall()

for row in rows:
    print(row)

connection.close()

(1, 'Product A', 150)
(2, 'Product B', 200)
(3, 'Product C', 300)
(4, 'Product D', 100)
(5, 'Product E', 50)
(6, 'Product F', 800)


In [12]:
db = SQLDatabase.from_uri("sqlite:///example.db")

print("Tables:", db.get_usable_table_names())

Tables: ['inventory']


In [13]:
print(db.get_table_info())


CREATE TABLE inventory (
	id INTEGER, 
	product_name TEXT NOT NULL, 
	stock_quantity INTEGER NOT NULL, 
	PRIMARY KEY (id)
)

/*
3 rows from inventory table:
id	product_name	stock_quantity
1	Product A	150
2	Product B	200
3	Product C	300
*/


In [14]:
llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0,
    max_tokens=512
)

print("✅ Groq LLM created successfully")

✅ Groq LLM created successfully


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

key = os.getenv("GROQ_API_KEY")

print("API key found:", key is not None)

In [22]:
from langchain_groq import ChatGroq
import os

llm = ChatGroq(
    model="qwen/qwen3.8-27b",
    temperature=0,
    max_tokens=512,
    api_key=os.getenv("GROQ_API_KEY")
)

print("✅ Fresh Groq LLM created")

✅ Fresh Groq LLM created


In [23]:
response = llm.invoke("Say hello in one short sentence.")

print(response.content)

Hello!


In [24]:
toolkit = SQLDatabaseToolkit(
    db=db,
    llm=llm
)

tools = toolkit.get_tools()

print("✅ SQL Toolkit recreated successfully")

for tool in tools:
    print("-", tool.name)

✅ SQL Toolkit recreated successfully
- sql_db_query
- sql_db_schema
- sql_db_list_tables
- sql_db_query_checker


In [25]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""You are an intelligent SQL database assistant.

Your job is to answer questions about the connected SQL database.

Rules:
1. Always inspect the database schema before writing SQL.
2. Use the available SQL tools to retrieve accurate information.
3. Never invent information that is not present in the database.
4. Remember the conversation history.
5. Use previous questions to understand follow-up questions.
6. Give clear and concise answers.
7. Format large numbers with commas.
8. If the requested information is not available, clearly tell the user.
"""
)

print("✅ SQL Agent recreated successfully")

✅ SQL Agent recreated successfully


In [26]:
query = "How many units of Product F are in stock?"

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": query
        }
    ]
})

print("🤖 Agent:", response["messages"][-1].content)

🤖 Agent: There are **800 units** of Product F in stock.


In [27]:
messages = []

print("✅ Conversation memory initialized")

✅ Conversation memory initialized


In [29]:
messages.append({
    "role": "user",
    "content": "How many units of Product F are in stock?"
})

response = agent.invoke({
    "messages": messages
})

messages = response["messages"]

print("🤖 Agent:", messages[-1].content)

🤖 Agent: There are **800 units** of Product F in stock.


In [31]:
messages.append({
    "role": "user",
    "content": "What about Product A?"
})

response = agent.invoke({
    "messages": messages
})

messages = response["messages"]

print("🤖 Agent:", messages[-1].content)

🤖 Agent: There are **150 units** of Product A in stock.


In [33]:
messages.append({
    "role": "user",
    "content": "What is the total stock of all products?"
})

response = agent.invoke({
    "messages": messages
})

messages = response["messages"]

print("🤖 Agent:", messages[-1].content)

🤖 Agent: The total stock of all products is **1,600 units**.


In [34]:
messages.append({
    "role": "user",
    "content": "Which products have more than 150 units in stock?"
})

response = agent.invoke({
    "messages": messages
})

messages = response["messages"]

print("🤖 Agent:", messages[-1].content)

🤖 Agent: The products with more than 150 units in stock are:

- **Product B**: 200 units
- **Product C**: 300 units
- **Product F**: 800 units


In [35]:
messages.append({
    "role": "user",
    "content": "Which product has the highest stock?"
})

response = agent.invoke({
    "messages": messages
})

messages = response["messages"]

print("🤖 Agent:", messages[-1].content)

🤖 Agent: **Product F** has the highest stock, with **800 units**.


In [36]:
messages.append({
    "role": "user",
    "content": "Show me all products with their stock quantities."
})

response = agent.invoke({
    "messages": messages
})

messages = response["messages"]

print("🤖 Agent:", messages[-1].content)

🤖 Agent: Here are all products with their stock quantities:

- **Product A**: 150 units
- **Product B**: 200 units
- **Product C**: 300 units
- **Product D**: 100 units
- **Product E**: 50 units
- **Product F**: 800 units


In [37]:
messages.append({
    "role": "user",
    "content": "How many units of Product Z are in stock?"
})

response = agent.invoke({
    "messages": messages
})

messages = response["messages"]

print("🤖 Agent:", messages[-1].content)

🤖 Agent: There is no record for **Product Z** in the inventory, so it is not in stock (or not tracked in this database).


In [38]:
print("\n" + "=" * 65)
print("          🤖 SQL DATABASE ASSISTANT")
print("=" * 65)

print("\nAsk questions about your database.")
print("Type 'exit' or 'quit' to stop.\n")

while True:

    if user_query.lower() in ["exit", "quit"]:
        print("\n🤖 Agent: Goodbye! 👋")
        break

    if not user_query:
        print("⚠️ Please enter a question.\n")
        continue

    messages.append({
        "role": "user",
        "content": user_query
    })

    try:
        response = agent.invoke({
            "messages": messages
        })

        messages = response["messages"]

        print("\n🤖 Agent:", messages[-1].content)
        print()



    except Exception as e:
        print("\n❌ Error:", str(e))
        print()


          🤖 SQL DATABASE ASSISTANT

Ask questions about your database.
Type 'exit' or 'quit' to stop.


🤖 Agent: **Product B** has **200 units** in stock.


🤖 Agent: Yes, that's correct. **Product B** has **200 units** in stock.


🤖 Agent: **Product C** has **300 units** in stock.


🤖 Agent: Yes, that's correct. **Product C** has **300 units** in stock.


🤖 Agent: **Product F** has the highest stock, with **800 units**.


🤖 Agent: Goodbye! 👋
